# Relaxation timing with NVIDIA ALCHEMI, on a platform GPU

Relaxes many strained crystals with MACE through the [NVIDIA ALCHEMI toolkit](https://github.com/NVIDIA/nvalchemi-toolkit)
in three ways, each its own workflow: **ALCHEMI batched** relaxes the structures together in batches on one GPU,
**ALCHEMI serial** and **ASE serial** relax them one at a time. A workflow reads the materials of its job and
publishes one timing row as `results.csv`.

This notebook saves the structures as materials in one set, creates the three workflows, runs one job per case,
and plots the structures relaxed per second against the number of structures.

## 1. Set up the environment and parameters
### 1.1. Install packages (JupyterLite)

In [ ]:
from mat3ra.notebooks_utils.packages import install_packages

await install_packages("made|api_examples")

### 1.2. Set parameters

In [ ]:
from datetime import datetime

from mat3ra.ide.compute import QueueName

# 2. Auth and organization parameters
ORGANIZATION_NAME = None  # set to use an organization as the owner, otherwise the personal account

# 3. Material parameters
MATERIAL_NAME = "Silicon"  # from Materials Standata
SUPERCELL_SCALING_FACTOR = [2, 2, 2]  # repetitions of the conventional cell
NUMBER_OF_STRUCTURES = 300
RANDOM_SEED = 0
STRAIN_STANDARD_DEVIATION = 0.01  # of each symmetric strain component
STRAIN_LIMIT = 0.02  # strain components are clipped to ±STRAIN_LIMIT
DISPLACEMENT_STANDARD_DEVIATION = 0.03  # Å, of each atomic coordinate
MATERIAL_SET_NAME = "ALCHEMI relaxation structures"

# 4. Workflow parameters
SCRIPT_FILE = "scripts/relaxation_alchemi.py"
REQUIREMENTS_FILE = "scripts/relaxation_alchemi_requirements.txt"
PYTHON_VERSION = "3.11.10"
PYTHON_BUILD = "Default"

# 5. Compute parameters
CLUSTER_NAME = "cluster-003"
QUEUE_NAME = QueueName.GOF  # H100 nodes on cluster-003 (GOF on-demand, GSF spot)
NODES = 1
PPN = 1
TIME_LIMIT = "01:00:00"

# 6. Job parameters: one job per (method, number of structures), on the first structures of the set
CASES = [
    ("ALCHEMI batched", 30), ("ALCHEMI batched", 50), ("ALCHEMI batched", 100), ("ALCHEMI batched", 300),
    ("ALCHEMI serial", 5), ("ALCHEMI serial", 10), ("ALCHEMI serial", 30),
    ("ASE serial", 5), ("ASE serial", 10), ("ASE serial", 30),
]
timestamp = datetime.now().strftime("%Y-%m-%d %H:%M")
POLL_INTERVAL = 30  # seconds

## 2. Authenticate and initialize API client
### 2.1. Authenticate

In [ ]:
from mat3ra.notebooks_utils.auth import authenticate


await authenticate()

### 2.2. Initialize API client

In [ ]:
from mat3ra.api_client import APIClient

client = APIClient.authenticate()

### 2.3. Select account

In [ ]:
selected_account = client.my_account

if ORGANIZATION_NAME:
    selected_account = client.get_account(name=ORGANIZATION_NAME)

ACCOUNT_ID = selected_account.id
print(f"✅ Selected account ID: {ACCOUNT_ID}, name: {selected_account.name}")

### 2.4. Select project

In [ ]:
projects = client.projects.list({"isDefault": True, "owner._id": ACCOUNT_ID})
project_id = projects[0]["_id"]
print(f"✅ Using project: {projects[0]['name']} ({project_id})")

## 3. Create materials
### 3.1. Build strained and displaced structures

Strained copies of a supercell of the crystal, with every atom displaced at random. Each job takes the first
structures of the set in the same order, so a structure is the same material in every job.

In [ ]:
import numpy as np
from mat3ra.esse.models.core.abstract.matrix_3x3 import Matrix3x3Schema
from mat3ra.made.material import Material
from mat3ra.made.tools.build_components.operations.core.modifications.perturb.helpers import create_maxwell_displacement
from mat3ra.made.tools.build_components.operations.core.modifications.strain.helpers import create_strain
from mat3ra.made.tools.helpers import create_supercell, get_material_with_conventional_lattice
from mat3ra.notebooks_utils.ipython.entity.material.visualize import visualize_materials as visualize
from mat3ra.standata.materials import Materials

crystal = get_material_with_conventional_lattice(Material.create(Materials.get_by_name_first_match(MATERIAL_NAME)))
supercell = create_supercell(crystal, scaling_factor=SUPERCELL_SCALING_FACTOR)
random_generator = np.random.default_rng(RANDOM_SEED)
materials = []
for index in range(NUMBER_OF_STRUCTURES):
    strain = random_generator.normal(0.0, STRAIN_STANDARD_DEVIATION, size=(3, 3))
    strain = np.clip((strain + strain.T) / 2, -STRAIN_LIMIT, STRAIN_LIMIT)
    strained = create_strain(supercell, strain_matrix=Matrix3x3Schema(root=(np.eye(3) + strain).tolist()))
    material = create_maxwell_displacement(
        strained, disorder_parameter=DISPLACEMENT_STANDARD_DEVIATION**2, random_seed=RANDOM_SEED + index, is_mass_used=False
    )
    material.name = f"{MATERIAL_NAME} {index}"
    materials.append(material)

visualize(materials[0])

### 3.2. Save materials to the platform in one set

In [ ]:
from mat3ra.notebooks_utils.core.entity.material.api import get_or_create_material

materials_set = client.materials.create_set({"name": f"{MATERIAL_SET_NAME} {timestamp}", "owner": {"_id": ACCOUNT_ID}})
saved_materials = [get_or_create_material(client, material, ACCOUNT_ID) for material in materials]
for saved_material in saved_materials:
    client.materials.move_to_set(saved_material["_id"], "", materials_set["_id"])

## 4. Create the workflows
### 4.1. Create one workflow per method

Each workflow reads its job's list of materials into `MATERIAL_IDS`, fetches those materials in two halves, and keeps
the `_id`, `name`, `basis` and `lattice` of each in `MATERIALS`. The relaxation script reads these two variables,
relaxes the materials in the job's order with its method, and publishes its timing row as `results.csv`.

In [ ]:
import copy
import re

from mat3ra.notebooks_utils.core.entity.workflow.api import set_execution_unit_input
from mat3ra.notebooks_utils.ipython.entity.workflow.visualize import visualize_workflow
from mat3ra.standata.workflows import WorkflowStandata
from mat3ra.wode.workflows import Workflow

job_materials_units = [
    {"type": "io", "subtype": "input", "source": "api", "name": "get-job", "flowchartId": "get-job", "head": True,
     "input": [{"type": "api", "endpoint": "jobs", "name": "DATA",
                "endpoint_options": {"params": {"query": "{'_id': JOB_ID}", "projection": "{}"}}}]},
    {"type": "assignment", "name": "assign-material-ids", "flowchartId": "assign-material-ids", "operand": "MATERIAL_IDS",
     "value": "[material['_id'] for material in DATA[0]['_materials']]", "input": [{"name": "DATA", "scope": "get-job"}]},
    {"type": "io", "subtype": "input", "source": "api", "name": "get-first-materials", "flowchartId": "get-first-materials",
     "input": [{"type": "api", "endpoint": "materials", "name": "FIRST_MATERIALS", "endpoint_options": {"params": {
         "query": "{'_id': {'$in': MATERIAL_IDS[:len(MATERIAL_IDS) // 2]}}", "projection": "{'limit': 500}"}}}]},
    {"type": "io", "subtype": "input", "source": "api", "name": "get-second-materials", "flowchartId": "get-second-materials",
     "input": [{"type": "api", "endpoint": "materials", "name": "SECOND_MATERIALS", "endpoint_options": {"params": {
         "query": "{'_id': {'$in': MATERIAL_IDS[len(MATERIAL_IDS) // 2:]}}", "projection": "{'limit': 500}"}}}]},
    {"type": "assignment", "name": "assign-materials", "flowchartId": "assign-materials", "operand": "MATERIALS",
     "value": "[{'_id': material['_id'], 'name': material['name'], 'basis': material['basis'], 'lattice': material['lattice']}"
              " for material in FIRST_MATERIALS + SECOND_MATERIALS]",
     "input": [{"name": "FIRST_MATERIALS", "scope": "get-first-materials"},
               {"name": "SECOND_MATERIALS", "scope": "get-second-materials"}]},
]
with open(SCRIPT_FILE) as file:
    script = file.read()
with open(REQUIREMENTS_FILE) as file:
    requirements = file.read()
custom_script_workflow = WorkflowStandata.filter_by_application("python").get_by_name_first_match("custom_script.json")

workflows = {}
for method in dict.fromkeys(method for method, _ in CASES):
    workflow_config = copy.deepcopy(custom_script_workflow)
    workflow_config.update(name=f"{method} relaxation", isMultiMaterial=True)
    workflow_config["units"][0]["name"] = workflow_config["name"]
    subworkflow = workflow_config["subworkflows"][0]
    relax_unit = subworkflow["units"][-1]
    relax_unit.update(name="relax", results=[{"name": "file_content", "filetype": "csv", "basename": "results.csv"}])
    for application in (subworkflow["application"], relax_unit["application"]):
        application.update(version=PYTHON_VERSION, build=PYTHON_BUILD)
    set_execution_unit_input(relax_unit, "script.py", re.sub(r"^METHOD = .*$", f'METHOD = "{method}"', script, flags=re.M))
    set_execution_unit_input(relax_unit, "requirements.txt", requirements)
    subworkflow.update(name=workflow_config["name"], units=copy.deepcopy(job_materials_units) + [relax_unit])
    for unit, next_unit in zip(subworkflow["units"], subworkflow["units"][1:]):
        unit["next"] = next_unit["flowchartId"]
    workflows[method] = Workflow.create(workflow_config)
    visualize_workflow(workflows[method])

### 4.2. Save workflows to the collection

In [ ]:
for workflow in workflows.values():
    saved_workflow = client.workflows.create(workflow.to_dict_without_special_keys(), owner_id=ACCOUNT_ID)
    print(f"✅ Workflow saved to collection: {saved_workflow['_id']}")

## 5. Create the compute configuration
### 5.1. Get list of clusters

In [ ]:
clusters = client.clusters.list()
print(f"Available clusters: {[cluster['hostname'] for cluster in clusters]}")

### 5.2. Create compute configuration for the jobs

In [ ]:
from mat3ra.ide.compute import Compute

cluster = next(cluster for cluster in clusters if CLUSTER_NAME in cluster["hostname"])
compute = Compute(cluster=cluster, queue=QUEUE_NAME, nodes=NODES, ppn=PPN, timeLimit=TIME_LIMIT)
print(f"Using cluster: {compute.cluster.hostname}, queue: {QUEUE_NAME}, nodes: {NODES}, ppn: {PPN}, time limit: {TIME_LIMIT}")

## 6. Create one job per case

In [ ]:
from mat3ra.notebooks_utils.job import create_job

job_ids = []
for method, number_of_structures in CASES:
    job = create_job(
        api_client=client,
        materials=saved_materials[:number_of_structures],
        workflow=workflows[method],
        project_id=project_id,
        owner_id=ACCOUNT_ID,
        prefix=f"{workflows[method].name} {number_of_structures} {timestamp}",
        compute=compute.to_dict(),
        materials_set=materials_set,
    )
    job_ids.append(job["_id"])
print(f"✅ Created {len(job_ids)} jobs: {job_ids}")

## 7. Submit the jobs and monitor the status

In [ ]:
from mat3ra.notebooks_utils.api.job import submit_jobs

submit_jobs(client.jobs, job_ids)

In [ ]:
from mat3ra.notebooks_utils.api.job import wait_for_jobs_to_finish_async

await wait_for_jobs_to_finish_async(client.jobs, job_ids, poll_interval=POLL_INTERVAL)

## 8. Results
### 8.1. Timing rows of all jobs

In [ ]:
import io

import pandas as pd
from mat3ra.notebooks_utils.io import read_from_url

tables = []
for job_id in job_ids:
    files = {file["key"].rsplit("/", 1)[-1]: file["signedUrl"] for file in client.jobs.list_files(job_id)}
    tables.append(pd.read_csv(io.StringIO(await read_from_url(files["results.csv"]))))
results = pd.concat(tables, ignore_index=True)
results

### 8.2. Structures relaxed per second

In [ ]:
import plotly.express as px

px.line(results.sort_values("structures"), x="structures", y="structures_per_second", color="method", markers=True)